In [91]:
import pandas as pd 
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from scipy import stats
from sklearn.inspection import permutation_importance
from sklearn.metrics import classification_report, confusion_matrix

In [92]:
df = pd.read_csv('ai_skepticism_dataset.csv')
df.shape 

(1000, 24)

In [93]:
df.columns

Index(['ai_model_name', 'query_category', 'ai_confidence_percentage',
       'response_character_count', 'has_cited_sources',
       'contains_hedging_words', 'includes_disclaimer', 'answer_detail_level',
       'respondent_age_bracket', 'education_level', 'digital_literacy_score',
       'ai_familiarity_level', 'decision_importance', 'urgency_level',
       'urgency_level_missing', 'belief_alignment_status',
       'belief_alignment_status_missing', 'subject_matter_expertise',
       'trust_score_out_of_10', 'performed_fact_check',
       'verification_duration_mins', 'answer_accuracy_percentage',
       'trust_calibration_valid', 'user_skepticism_category'],
      dtype='str')

In [94]:
# What are the factors that determine whether a person trusts AI or not?
 
"""
Analysis approach:
Step 1: Select predictor variable (user_skepticism_category)
- Most human-interpretable and maps directly to distinct trust profiles (Blind Trust / Moderate Trust / Skeptical / Highly Skeptical)
- 'trust_score_out_of_10' distribution is heavily left-skewed (mean 7.9, median 8.2, max 10.0). Most people score high. This means a regression on the raw score may produce a model that always predicts "moderately high trust" and still looks accurate — a misleading result.

Step 2: Clean data and check variables that need encoding
- Handle missing values such as urgency_level and belief_alignment_status (impute with mode and create a new column to signal values that were missing); fact_check_method_used (remove column)

Step 3: Encode categoricals 
- One-hot encoding for columns with no inherent order: ai_model_name, query_category.
- Ordinal encoding for clear, defensible ranked order: answer_detail_level, respondent_age_bracket, education_level, digital_literacy_score, ai_familiarity_level,  decision_importance, urgency_level.

Step 4: Perform preliminary analysis and remove data leakage variables
- Correlation analysis 
- Remove variables that could potentially cause leakage
- If 'performed_fact_check' and 'verification_duration_mins' are kept as features, the model is answering: “Given everything we know about this interaction including what happened afterwards, what trust category does the person fall into?” — which is a descriptive profiling exercise, not a predictive one.
- If they are removed as features, the model answers: “Based only on what we knew before and during the interaction, can we predict how much someone will trust the AI?” — which is a genuinely predictive framing.

Step 5: Modelling 
- Generate statistical table 
- Establish train/test split (70/30) for variable importance and testing predictive accuracy 
- Produce feature importance scores, which directly answer your research question: which variables contribute most to predicting trust? 
- Accuracy of Random Forest vs Logistic Regression 

Step 6: Interpret and Communicate the Findings
- A feature can be important in the random forest but its direction (does more familiarity increase or decrease trust?) only comes from the regression.
- Group the findings back into the feature groups from Step 2 (person-level, response-level, situational, behavioural). A structured finding like "response characteristics matter more than respondent demographics in predicting trust" is more actionable than a raw ranked list.
- Acknowledge the dataset's limits. 1,000 rows is a reasonable learning exercise, but thin for a four-class model with 20+ features — especially the Highly Skeptical class (57 rows). Any published conclusions should be framed as exploratory.

"""

'\nAnalysis approach:\nStep 1: Select predictor variable (user_skepticism_category)\n- Most human-interpretable and maps directly to distinct trust profiles (Blind Trust / Moderate Trust / Skeptical / Highly Skeptical)\n- \'trust_score_out_of_10\' distribution is heavily left-skewed (mean 7.9, median 8.2, max 10.0). Most people score high. This means a regression on the raw score may produce a model that always predicts "moderately high trust" and still looks accurate — a misleading result.\n\nStep 2: Clean data and check variables that need encoding\n- Handle missing values such as urgency_level and belief_alignment_status (impute with mode and create a new column to signal values that were missing); fact_check_method_used (remove column)\n\nStep 3: Encode categoricals \n- One-hot encoding for columns with no inherent order: ai_model_name, query_category.\n- Ordinal encoding for clear, defensible ranked order: answer_detail_level, respondent_age_bracket, education_level, digital_liter

In [95]:
# Step 2: Clean data 

missing_counts = df.isnull().sum()
print(missing_counts)

ai_model_name                      0
query_category                     0
ai_confidence_percentage           0
response_character_count           0
has_cited_sources                  0
contains_hedging_words             0
includes_disclaimer                0
answer_detail_level                0
respondent_age_bracket             0
education_level                    0
digital_literacy_score             0
ai_familiarity_level               0
decision_importance                0
urgency_level                      0
urgency_level_missing              0
belief_alignment_status            0
belief_alignment_status_missing    0
subject_matter_expertise           0
trust_score_out_of_10              0
performed_fact_check               0
verification_duration_mins         0
answer_accuracy_percentage         0
trust_calibration_valid            0
user_skepticism_category           0
dtype: int64


In [96]:
df.dtypes

ai_model_name                          str
query_category                         str
ai_confidence_percentage           float64
response_character_count             int64
has_cited_sources                     bool
contains_hedging_words                bool
includes_disclaimer                   bool
answer_detail_level                    str
respondent_age_bracket                 str
education_level                        str
digital_literacy_score                 str
ai_familiarity_level                   str
decision_importance                    str
urgency_level                          str
urgency_level_missing                 bool
belief_alignment_status               bool
belief_alignment_status_missing       bool
subject_matter_expertise              bool
trust_score_out_of_10              float64
performed_fact_check                  bool
verification_duration_mins         float64
answer_accuracy_percentage         float64
trust_calibration_valid               bool
user_skepti

In [97]:
# Step 3a: Encode categoricals 
# One-hot encoding for nominal variables: ai_model_name, query_category.

from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False, drop='first') # Drops the first category alphabetically for each feature to create K-1 dummy columns instead of K to prevent multicollinearity.
encoded_array = ohe.fit_transform(df[['ai_model_name', 'query_category']]) 

encoded_cols = ohe.get_feature_names_out(['ai_model_name', 'query_category'])
encoded_df = pd.DataFrame(encoded_array, columns=encoded_cols, index=df.index) # Convert the encoded array to a DataFrame for joining with the original dataset
df = df.join(encoded_df) # Join the new encoded columns

print(encoded_cols)
print(encoded_df)

['ai_model_name_Claude' 'ai_model_name_GPT-4' 'ai_model_name_Gemini'
 'ai_model_name_Llama' 'ai_model_name_Mistral'
 'query_category_current_events' 'query_category_factual_historical'
 'query_category_financial_advice' 'query_category_general_knowledge'
 'query_category_legal_advice' 'query_category_math_calculation'
 'query_category_medical_advice' 'query_category_opinion_based'
 'query_category_recipe_cooking' 'query_category_scientific_facts'
 'query_category_technical_coding']
     ai_model_name_Claude  ai_model_name_GPT-4  ai_model_name_Gemini  \
0                     1.0                  0.0                   0.0   
1                     0.0                  0.0                   0.0   
2                     1.0                  0.0                   0.0   
3                     1.0                  0.0                   0.0   
4                     1.0                  0.0                   0.0   
..                    ...                  ...                   ...   
995      

In [107]:
# After one-hot encoding, check which original categories exist in feature_columns
original_categories = df['ai_model_name'].unique()
print("All categories:", original_categories)

encoded_cols = [c for c in feature_columns if c.startswith('ai_model_name_')]
print("Encoded columns kept:", encoded_cols)

# The reference = any category NOT in the encoded columns
kept = [c.replace('ai_model_name_', '') for c in encoded_cols]
reference = [c for c in original_categories if c not in kept]
print("Reference (dropped) category:", reference)

All categories: <ArrowStringArray>
['Claude', 'Llama', 'Mistral', 'ChatGPT-3.5', 'Gemini', 'GPT-4']
Length: 6, dtype: str
Encoded columns kept: ['ai_model_name_Claude', 'ai_model_name_GPT-4', 'ai_model_name_Gemini', 'ai_model_name_Llama', 'ai_model_name_Mistral']
Reference (dropped) category: ['ChatGPT-3.5']


In [108]:
original_categories = df['query_category'].unique()
encoded_cols = [c for c in feature_columns if c.startswith('query_category_')]
kept = [c.replace('query_category_', '') for c in encoded_cols]
reference = [c for c in original_categories if c not in kept]
print("Reference query category:", reference)

Reference query category: ['creative_writing']


In [98]:
# Step 3b: Encode categoricals 
# Ordinal encoding: answer_detail_level, respondent_age_bracket, education_level, digital_literacy_score, ai_familiarity_level,  decision_importance, urgency_level
answer_detail_map = {
    'Vague'        : 1,
    'Moderate'     : 2,
    'Specific'     : 3,
    'Very Specific': 4
}

age_bracket_map = {
    '18-24': 1,
    '25-34': 2,
    '35-44': 3,
    '45-54': 4,
    '55-64': 5,
    '65+'  : 6
}

education_map = {
    'High School' : 1,
    'Bachelors'   : 2,
    'Masters'     : 3,
    'Professional': 4,
    'PhD'         : 5
}

digital_literacy_map = {
    'Low'   : 1,
    'Medium': 2,
    'High'  : 3,
    'Expert': 4
}

ai_familiarity_map = {
    'First Time'  : 1,
    'Beginner'    : 2,
    'Intermediate': 3,
    'Advanced'    : 4,
    'Expert'      : 5
}

decision_importance_map = {
    'Low'     : 1,
    'Medium'  : 2,
    'High'    : 3,
    'Critical': 4
}

urgency_map = {
    'Low'   : 1,
    'Medium': 2,
    'High'  : 3
}

df['answer_detail_level_encoded']    = df['answer_detail_level'].map(answer_detail_map)
df['respondent_age_bracket_encoded'] = df['respondent_age_bracket'].map(age_bracket_map)
df['education_level_encoded']        = df['education_level'].map(education_map)
df['digital_literacy_score_encoded'] = df['digital_literacy_score'].map(digital_literacy_map)
df['ai_familiarity_level_encoded']   = df['ai_familiarity_level'].map(ai_familiarity_map)
df['decision_importance_encoded']    = df['decision_importance'].map(decision_importance_map)
df['urgency_level_encoded']          = df['urgency_level'].map(urgency_map)

# additional step to check that the variables are encoded correctly 
encoded_pairs = [
    ('answer_detail_level',    'answer_detail_level_encoded'),
    ('respondent_age_bracket', 'respondent_age_bracket_encoded'),
    ('education_level',        'education_level_encoded'),
    ('digital_literacy_score', 'digital_literacy_score_encoded'),
    ('ai_familiarity_level',   'ai_familiarity_level_encoded'),
    ('decision_importance',    'decision_importance_encoded'),
    ('urgency_level',          'urgency_level_encoded')
]

for orig, enc in encoded_pairs:
    print(f"=== {orig} ===")
    mapping_check = df[[orig, enc]].drop_duplicates().sort_values(enc) # Removes all repeated row combinations
    print(mapping_check.to_string(index=False)) # The index=False argument suppresses the row numbers (0, 1, 2, 3) that pandas normally prints on the left side
    print(f"  Nulls in encoded column: {df[enc].isna().sum()}")
    print()

=== answer_detail_level ===
answer_detail_level  answer_detail_level_encoded
              Vague                            1
           Moderate                            2
           Specific                            3
      Very Specific                            4
  Nulls in encoded column: 0

=== respondent_age_bracket ===
respondent_age_bracket  respondent_age_bracket_encoded
                 18-24                               1
                 25-34                               2
                 35-44                               3
                 45-54                               4
                 55-64                               5
                   65+                               6
  Nulls in encoded column: 0

=== education_level ===
education_level  education_level_encoded
    High School                        1
      Bachelors                        2
        Masters                        3
   Professional                        4
            PhD       

In [99]:
df.columns

Index(['ai_model_name', 'query_category', 'ai_confidence_percentage',
       'response_character_count', 'has_cited_sources',
       'contains_hedging_words', 'includes_disclaimer', 'answer_detail_level',
       'respondent_age_bracket', 'education_level', 'digital_literacy_score',
       'ai_familiarity_level', 'decision_importance', 'urgency_level',
       'urgency_level_missing', 'belief_alignment_status',
       'belief_alignment_status_missing', 'subject_matter_expertise',
       'trust_score_out_of_10', 'performed_fact_check',
       'verification_duration_mins', 'answer_accuracy_percentage',
       'trust_calibration_valid', 'user_skepticism_category',
       'ai_model_name_Claude', 'ai_model_name_GPT-4', 'ai_model_name_Gemini',
       'ai_model_name_Llama', 'ai_model_name_Mistral',
       'query_category_current_events', 'query_category_factual_historical',
       'query_category_financial_advice', 'query_category_general_knowledge',
       'query_category_legal_advice', 'query

In [100]:
# Step 4a: Perform preliminary analysis and remove data leakage variables
# Boolean variables — yes, directly. Python treats True as 1 and False as 0, so Pearson correlation works on boolean columns without any conversion.

# ---------------------------------------------------------------
# STEP 1: Encode user_skepticism_category as ordinal numeric
# Trust increases from Highly Skeptical → Blind Trust
# This ordinal encoding allows correlation coefficients to reflect whether a feature pushes toward more or less trust (as opposed to label encoder which does not care about real ordering)
# ---------------------------------------------------------------
trust_order = {
    'Highly Skeptical': 1,
    'Skeptical'       : 2,
    'Moderate Trust'  : 3,
    'Blind Trust'     : 4
}

df['trust_encoded'] = df['user_skepticism_category'].map(trust_order)

# ---------------------------------------------------------------
# STEP 2: Define feature columns
# Boolean columns converted to int for correlation computation
# ---------------------------------------------------------------
feature_columns = [
    'ai_confidence_percentage', 'answer_accuracy_percentage', 'trust_calibration_valid', 'belief_alignment_status', 'performed_fact_check', 'verification_duration_mins',
    'response_character_count', 'has_cited_sources',
    'contains_hedging_words', 'includes_disclaimer',
    'urgency_level_missing', 
    'belief_alignment_status_missing', 'subject_matter_expertise',
    'ai_model_name_Claude', 'ai_model_name_GPT-4', 'ai_model_name_Gemini',
    'ai_model_name_Llama', 'ai_model_name_Mistral',
    'query_category_current_events', 'query_category_factual_historical',
    'query_category_financial_advice', 'query_category_general_knowledge',
    'query_category_legal_advice', 'query_category_math_calculation',
    'query_category_medical_advice', 'query_category_opinion_based',
    'query_category_recipe_cooking', 'query_category_scientific_facts',
    'query_category_technical_coding',
    'answer_detail_level_encoded',
    'respondent_age_bracket_encoded', 'education_level_encoded',
    'digital_literacy_score_encoded', 'ai_familiarity_level_encoded',
    'decision_importance_encoded', 'urgency_level_encoded'
]

# Convert boolean columns to int (True=1, False=0)
X = df[feature_columns].astype(float)

# ---------------------------------------------------------------
# STEP 3: Compute Pearson correlation and p-value
# for each feature against the encoded trust score
#
# Pearson r interpretation:
#   +1.0       = perfect positive correlation
#    0.0       = no linear relationship
#   -1.0       = perfect negative correlation
#
# Strength thresholds (Cohen, 1988):
#   |r| < 0.10  = negligible
#   |r| 0.10–0.29 = weak
#   |r| 0.30–0.49 = moderate
#   |r| >= 0.50   = strong
#
# p-value < 0.05 = statistically significant at 95% level
# ---------------------------------------------------------------
results = []

for feature in feature_columns:
    r, p = stats.pearsonr(X[feature], df['trust_encoded'])
    results.append({
        'Feature'        : feature,
        'Correlation (r)': round(r, 4),
        'P-value'        : round(p, 4),
        'Significant'    : 'Yes' if p < 0.05 else 'No',
        'Direction'      : 'Positive' if r > 0 else 'Negative',
        'Strength'       : 'Strong'     if abs(r) >= 0.50 else
                           'Moderate'   if abs(r) >= 0.30 else
                           'Weak'       if abs(r) >= 0.10 else
                           'Negligible'
    })

# ---------------------------------------------------------------
# STEP 4: Sort by absolute correlation strength (strongest first)
# ---------------------------------------------------------------
corr_df = pd.DataFrame(results)
corr_df = corr_df.sort_values('Correlation (r)', key=abs, ascending=False).reset_index(drop=True)

print("="*70)
print("CORRELATION ANALYSIS — Features vs user_skepticism_category")
print("Encoding: Highly Skeptical=1, Skeptical=2, Moderate Trust=3, Blind Trust=4")
print("Positive r = feature associated with higher trust (toward Blind Trust)")
print("Negative r = feature associated with lower trust (toward Highly Skeptical)")
print("="*70)
print(corr_df.to_string(index=False))

# ---------------------------------------------------------------
# STEP 5: Print summary by significance and strength
# ---------------------------------------------------------------
sig_df = corr_df[corr_df['Significant'] == 'Yes']
print(f"\n{'='*70}")
print(f"SIGNIFICANT CORRELATIONS ONLY (p < 0.05) — {len(sig_df)} of {len(corr_df)} features")
print(f"{'='*70}")
print(sig_df.to_string(index=False))

CORRELATION ANALYSIS — Features vs user_skepticism_category
Encoding: Highly Skeptical=1, Skeptical=2, Moderate Trust=3, Blind Trust=4
Positive r = feature associated with higher trust (toward Blind Trust)
Negative r = feature associated with lower trust (toward Highly Skeptical)
                          Feature  Correlation (r)  P-value Significant Direction   Strength
             performed_fact_check          -0.7073   0.0000         Yes  Negative     Strong
         ai_confidence_percentage           0.5429   0.0000         Yes  Positive     Strong
       verification_duration_mins          -0.5195   0.0000         Yes  Negative     Strong
       answer_accuracy_percentage           0.4675   0.0000         Yes  Positive   Moderate
      decision_importance_encoded          -0.3130   0.0000         Yes  Negative   Moderate
          belief_alignment_status           0.2857   0.0000         Yes  Positive       Weak
  query_category_financial_advice          -0.2381   0.0000         

In [101]:
# Step 4b: Perform preliminary analysis and remove data leakage variables
# Exclude data leakage columns ('trust_score_out_of_10', 'answer_accuracy_percentage', 'trust_calibration_valid', 'performed_fact_check', 'verification_duration_mins') 
# Exclude original columns that were encoded ('ai_model_name', 'query_category')

feature_columns = [
    'ai_confidence_percentage', 'belief_alignment_status',
    'response_character_count', 'has_cited_sources',
    'contains_hedging_words', 'includes_disclaimer',
    'urgency_level_missing', 
    'belief_alignment_status_missing', 'subject_matter_expertise',
    'ai_model_name_Claude', 'ai_model_name_GPT-4', 'ai_model_name_Gemini',
    'ai_model_name_Llama', 'ai_model_name_Mistral',
    'query_category_current_events', 'query_category_factual_historical',
    'query_category_financial_advice', 'query_category_general_knowledge',
    'query_category_legal_advice', 'query_category_math_calculation',
    'query_category_medical_advice', 'query_category_opinion_based',
    'query_category_recipe_cooking', 'query_category_scientific_facts',
    'query_category_technical_coding',
    'answer_detail_level_encoded',
    'respondent_age_bracket_encoded', 'education_level_encoded',
    'digital_literacy_score_encoded', 'ai_familiarity_level_encoded',
    'decision_importance_encoded', 'urgency_level_encoded'
]

In [102]:
# Step 5a: Modelling 
# Generate statistical table 

# ---------------------------------------------------------------
# STEP 1: Define X and Y variables
# ---------------------------------------------------------------

X = df[feature_columns].astype(float)
y = df['user_skepticism_category']

# ---------------------------------------------------------------
# STEP 2: Scale features on full dataset (so that coefficients lie on the same scale)
# ---------------------------------------------------------------
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)

# ---------------------------------------------------------------
# STEP 3: Add constant (intercept)
# ---------------------------------------------------------------
X_sm = sm.add_constant(X_sc)

# ---------------------------------------------------------------
# STEP 4: Encode target variable as a class label, not a point on a scale (i.e. ordinal)
# ---------------------------------------------------------------
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Class encoding (0 = baseline reference category):")
for i, cls in enumerate(le.classes_):
    marker = " <-- BASELINE" if i == 0 else ""
    print(f"  {i} = {cls}{marker}")

# ---------------------------------------------------------------
# STEP 5: Fit MNLogit without freq_weights
# freq_weights breaks covariance matrix computation in statsmodels
# Use method='bfgs' which is robust to near-singular matrices
# The summary() method requires a valid covariance matrix to compute standard errors, z-scores and p-values
# ---------------------------------------------------------------
mnlogit_model  = sm.MNLogit(y_encoded, X_sm)
mnlogit_result = mnlogit_model.fit(
    method='bfgs',
    maxiter=1000,
    disp=False    # freq_weights removed — fixes ValueError
)

# ---------------------------------------------------------------
# STEP 6: Print summary table
# ---------------------------------------------------------------
print("\n" + "="*70)
print("MULTINOMIAL LOGISTIC REGRESSION — FULL DATASET (n=1,000)")
print("Baseline / Reference Category: Blind Trust")
print("="*70)

print(mnlogit_result.summary(
    xname=['const'] + feature_columns,
    alpha=0.05
))

Class encoding (0 = baseline reference category):
  0 = Blind Trust <-- BASELINE
  1 = Highly Skeptical
  2 = Moderate Trust
  3 = Skeptical

MULTINOMIAL LOGISTIC REGRESSION — FULL DATASET (n=1,000)
Baseline / Reference Category: Blind Trust
                          MNLogit Regression Results                          
Dep. Variable:                      y   No. Observations:                 1000
Model:                        MNLogit   Df Residuals:                      901
Method:                           MLE   Df Model:                           96
Date:                Sat, 10 Oct 2026   Pseudo R-squ.:                  0.4821
Time:                        20:29:17   Log-Likelihood:                -645.49
converged:                       True   LL-Null:                       -1246.3
Covariance Type:            nonrobust   LLR p-value:                1.932e-190
                              y=1       coef    std err          z      P>|z|      [0.025      0.975]
------------------------

In [106]:
# Extract mean and standard deviation per feature
# Because your features were standardised using StandardScaler before fitting, each coefficient represents the effect of a one standard deviation increase in that feature — not a one-unit increase. 
scaler_stats = pd.DataFrame({
    'Feature'  : feature_columns,
    'Mean'     : scaler.mean_.round(2),
    'Std Dev'  : scaler.scale_.round(2)
}).sort_values('Std Dev', ascending=False)

print(scaler_stats.to_string(index=False))

                          Feature   Mean  Std Dev
         response_character_count 429.02   221.53
         ai_confidence_percentage  72.01    14.66
   respondent_age_bracket_encoded   3.54     1.68
          education_level_encoded   3.04     1.43
     ai_familiarity_level_encoded   2.99     1.41
   digital_literacy_score_encoded   2.49     1.13
      answer_detail_level_encoded   2.46     1.11
      decision_importance_encoded   2.46     1.11
            urgency_level_encoded   1.99     0.72
         subject_matter_expertise   0.48     0.50
                has_cited_sources   0.52     0.50
           contains_hedging_words   0.51     0.50
              includes_disclaimer   0.51     0.50
          belief_alignment_status   0.68     0.47
  belief_alignment_status_missing   0.34     0.47
            urgency_level_missing   0.22     0.41
             ai_model_name_Gemini   0.19     0.39
             ai_model_name_Claude   0.17     0.38
              ai_model_name_Llama   0.16     0.37


In [103]:
# Step 5b: Modelling 
# Train/test split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y) # stratify=y ensures every class is proportionally represented in both sets 

In [104]:
# Step 5c: Modelling 
# Produce variable importance

rf = RandomForestClassifier(n_estimators=100, random_state=42) # specifies the number of individual decision trees (i.e. bootstrap samples) that will be built in the Random Forest ensemble
rf.fit(X_train, y_train)

# Although a Random Forest model naturally generates Out-Of-Bag (OOB) samples during training, standard implementations like Scikit-Learn's permutation_importance() require an explicit holdout dataset (X_test, y_test) to compute out-of-sample score drops accurately.
perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42)
# n_repeats: For every feature column in X_test, the function randomly shuffles (permutes) that column's values 10 separate times while keeping all other columns and y_test untouched.

perm_df = pd.DataFrame({
    'Feature'   : feature_columns,
    'Importance': perm.importances_mean
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print(f"\n--- Permutation Feature Importances (Random Forest) ---")
print(f"Top 10 features most predictive of AI trust:\n")
print(perm_df.head(10).to_string(index=False))

# Each one-hot column is a binary flag i.e. 1 means it comes from GPT-4 and 0 did not come from GPT-4; GPT-4 specifically produces a trust pattern that is distinct from all other models combined


--- Permutation Feature Importances (Random Forest) ---
Top 10 features most predictive of AI trust:

                       Feature  Importance
      ai_confidence_percentage    0.157333
   decision_importance_encoded    0.064333
       belief_alignment_status    0.063667
             has_cited_sources    0.039000
  ai_familiarity_level_encoded    0.022000
digital_literacy_score_encoded    0.019333
           ai_model_name_GPT-4    0.016667
          ai_model_name_Claude    0.015000
   query_category_legal_advice    0.014333
      response_character_count    0.013333


In [105]:
# Step 5d: Modelling 
# Random Forest's accuracy 

rf_preds = rf.predict(X_test)

print(f"\n--- Random Forest ---")
print(f"Accuracy : {accuracy_score(y_test, rf_preds):.2f}")
print(f"\nClassification Report:")
print(classification_report(y_test, rf_preds))
print(f"Confusion Matrix:")
print(confusion_matrix(y_test, rf_preds,
      labels=['Blind Trust', 'Moderate Trust', 'Skeptical', 'Highly Skeptical']))


--- Random Forest ---
Accuracy : 0.60

Classification Report:
                  precision    recall  f1-score   support

     Blind Trust       0.61      0.46      0.52        79
Highly Skeptical       1.00      0.06      0.11        17
  Moderate Trust       0.57      0.65      0.60       110
       Skeptical       0.63      0.77      0.69        94

        accuracy                           0.60       300
       macro avg       0.70      0.48      0.48       300
    weighted avg       0.62      0.60      0.58       300

Confusion Matrix:
[[36 36  7  0]
 [19 71 20  0]
 [ 4 18 72  0]
 [ 0  0 16  1]]


In [78]:
# Step 5d: Modelling 
# Logistic Regression's accuracy

X_train_sc   = scaler.fit_transform(X_train)  
X_test_sc    = scaler.transform(X_test)      

lr = LogisticRegression(max_iter=1000, random_state=42) 
# max_iter=1000 is an iterative algorithm that gradually adjusts the coefficients in small steps, converging toward the best solution over multiple iterations.
lr.fit(X_train_sc, y_train)
lr_pred = lr.predict(X_test_sc)

print(f"\n--- Logistic Regression ---")
print(f"Accuracy : {accuracy_score(y_test, lr_pred):.2f}")
print(f"\nClassification Report:")
print(classification_report(y_test, lr_pred))
print(f"Confusion Matrix:")
print(confusion_matrix(y_test, lr_pred))


--- Logistic Regression ---
Accuracy : 0.58

Classification Report:
                  precision    recall  f1-score   support

     Blind Trust       0.61      0.46      0.52        79
Highly Skeptical       0.36      0.29      0.32        17
  Moderate Trust       0.53      0.63      0.57       110
       Skeptical       0.66      0.68      0.67        94

        accuracy                           0.58       300
       macro avg       0.54      0.51      0.52       300
    weighted avg       0.58      0.58      0.58       300

Confusion Matrix:
[[36  0 40  3]
 [ 0  5  0 12]
 [23  0 69 18]
 [ 0  9 21 64]]
